# Role-Aware Podcast Summarization with Pseudo-Label Supervision

This notebook contains the full end-to-end pipeline for our final project:
1. SPoRC dataset preprocessing
2. Two-stage extractive–abstractive summarization
3. Role-aware summary generation (host / guest)
4. Evaluation and conflict analysis

All experiments are conducted on the SPoRC dataset.


### Step 0: Setup of Environment

Here we first import all necessary libraries and set the device to either CPU or GPU. <br>
We then define the project directory where all datasets, intermediate files, and final outputs will be stored. <br>
If the directory does not exist, it will be created automatically.

In [ ]:
import os
import gc
import pandas as pd
import numpy as np
import gzip
import json
import torch
import torch.nn as nn
import torch.nn.functional as F
import nltk
import random
import re

from tqdm import tqdm
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModel, BartForConditionalGeneration, AutoModelForSequenceClassification
from sentence_transformers import SentenceTransformer, util
from collections import defaultdict

!pip install rouge-score
from rouge_score import rouge_scorer

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# Define working directory
PROJECT_DIR = "/Users/allin1307/Desktop/semester 3/NLP/project"     # Note that this would need to be changed to match own path
os.makedirs(PROJECT_DIR, exist_ok=True)
print(f"Working directory: {PROJECT_DIR}")

## 1. Dataset Preprocessing

We preprocess the SPoRC dataset by downloading it from Hugging Face, converting speaker-turn data into CSV format, filtering speaker roles, and removing empty or low-quality utterances. <br>
This step ensures clean, role-consistent input for downstream summarization.

### Step 1: Download dataset from Hugging Face (auto local check)

This step downloads the SPoRC dataset from Hugging Face if local copies are not already present. The dataset is stored as gzipped JSONL files for both episode-level metadata and speaker-turn transcripts.<br>
If the files already exist locally, the download step is skipped to ensure reproducibility and avoid redundant network calls.

In [ ]:
episodes_path = os.path.join(PROJECT_DIR, "episodeLevelData.jsonl.gz")
turns_path = os.path.join(PROJECT_DIR, "speakerTurnData.jsonl.gz")

# --- Check if local files exist ---
if os.path.exists(episodes_path) and os.path.exists(turns_path):
    print(f"Found existing dataset files, skipping download.\n"
          f"Using local copies:\n{episodes_path}\n{turns_path}")
else:
    print("Local files not found. Downloading from Hugging Face...")
    dataset = load_dataset("blitt/SPoRC")
    dataset["episodes"].to_json(episodes_path, orient="records", lines=True)
    dataset["turns"].to_json(turns_path, orient="records", lines=True)
    print(f"Downloaded to:\n{episodes_path}\n{turns_path}")

### Step 2: Decompress gzipped JSONL files into plain .jsonl

The dataset is stored in `.jsonl.gz` format.<br>
We decompress the files so that subsequent reading and conversion steps are faster.

In [ ]:
#episodes_jsonl = episodes_path.replace(".gz", "")
turns_jsonl = turns_path.replace(".gz", "")

def decompress_gz(gz_path):
    jsonl_path = gz_path.replace(".gz", "")
    with gzip.open(gz_path, "rb") as f_in, open(jsonl_path, "wb") as f_out:
        f_out.write(f_in.read())
    print(f"Decompressed: {os.path.basename(jsonl_path)}")
    return jsonl_path

if os.path.exists(turns_jsonl):
    print(f"Found decompressed JSONL files, skipping decompression.")
else:
    episodes_jsonl = decompress_gz(episodes_path)
    turns_jsonl = decompress_gz(turns_path)

### Step 3: Preview structure of speaker-turn-level data

We inspect the first few lines of the speaker-turn-level data to understand available columns and ensure data integrity.

In [ ]:
turns_gz_path = os.path.join(PROJECT_DIR, turns_path)
print("Previewing file:", turns_gz_path)

# Read first 2 JSON lines from the compressed file
with gzip.open(turns_gz_path, "rt", encoding="utf-8") as f:
    first_lines = []
    for i in range(2):
        try:
            first_lines.append(json.loads(next(f)))
        except StopIteration:
            break
        except json.JSONDecodeError as e:
            print("JSON decode error on line", i, ":", e)
            break

print("Columns:", list(first_lines[0].keys()))
display(pd.DataFrame(first_lines))

### Step 4: Convert speakerTurnData.jsonl.gz to CSV
Converts the compressed speaker-turn-level JSONL file to CSV format.<br>
This step simplifies downstream processing and filtering using pandas.

In [ ]:
csv_path = os.path.join(PROJECT_DIR, "sporc_turns_selected.csv")

if os.path.exists(csv_path):
    print("Found existing CSV file, skipping conversion.")
    print("Using:", csv_path)
else:
    # Read directly from compressed JSONL.GZ to save memory
    print("Converting speakerTurnData.jsonl.gz to CSV...")
    with gzip.open(os.path.join(PROJECT_DIR, "speakerTurnData.jsonl.gz"), "rt", encoding="utf-8") as f_in:
        lines = [json.loads(line) for line in tqdm(f_in, desc="Converting")]
        df_turns = pd.DataFrame(lines)
        df_turns.to_csv(csv_path, index=False)
    print("Saved as CSV:", csv_path)
    del df_turns
    gc.collect()

### Step 5: Check and display role counts
We examine the distribution of inferred speaker roles (e.g. *host*, *guest*, etc.) to identify which labels to keep and which to filter out.

In [ ]:
turns_gz_path = os.path.join(PROJECT_DIR, turns_path)
print("Reading from:", turns_gz_path)

# Load inferredSpeakerRole column only
roles = []
with gzip.open(turns_gz_path, "rt", encoding="utf-8") as f:
    for i, line in enumerate(f):
        record = json.loads(line)
        roles.append(record.get("inferredSpeakerRole", None))
        if i > 200000:  # limit to first 200k lines for speed
            break

df_roles = pd.DataFrame(roles, columns=["inferredSpeakerRole"])
role_counts = df_roles["inferredSpeakerRole"].value_counts(dropna=False)

print("Role distribution BEFORE filtering:\n", role_counts)
print("\nUnique role labels:", df_roles["inferredSpeakerRole"].unique())

### Step 6: Filter host/guest roles only
Filters the dataset to include only *host* and *guest* speaker roles.<br>
Removes other categories that are irrelevant for conversational analysis.

In [ ]:
out_filtered = os.path.join(PROJECT_DIR, "sporc_turns_selected_clean.csv")

if os.path.exists(out_filtered):
    print("Found existing filtered file:", out_filtered)
    filtered = pd.read_csv(out_filtered)
    role_col = "inferredSpeakerRole" if "inferredSpeakerRole" in filtered.columns else "role"
    print("\nRole distribution AFTER filtering:\n", filtered[role_col].value_counts())
else:
    df_full = pd.read_csv(csv_path)

    # Keep only host / guest
    keep_labels = {"host", "guest"}
    filtered = df_full[df_full["inferredSpeakerRole"].isin(keep_labels)].copy()
    removed = len(df_full) - len(filtered)

    filtered.to_csv(out_filtered, index=False)
    print(f"Filtered roles: kept {len(filtered):,}, removed {removed:,}")
    print(f"Saved → {out_filtered}")


### Step 7: Count empty and short (<10 chars) utterances
Counts empty or very short text utterances to assess the need for cleaning.

In [ ]:
empty_count = filtered["text"].isna().sum()
short_count = (filtered["text"].fillna("").str.len() < 10).sum()
print(f"Total: {len(filtered):,}")
print(f"Empty utterances: {empty_count:,}")
print(f"Short utterances (<10 chars): {short_count:,}")

### Step 8: Remove empty and meaningless short texts
Final cleaning step: removes empty, whitespace-only, punctuation-only, and semantically meaningless short utterances.<br>
Ensures only substantive conversational content remains for downstream tasks (e.g., topic modeling or sentiment analysis).

In [ ]:
df = filtered.copy()

# Remove NaN / empty / whitespace-only
df = df.dropna(subset=["text"])
df["text"] = df["text"].astype(str).str.strip()
df = df[df["text"] != ""]
df = df[~df["text"].str.fullmatch(r"[\.,!\?\-–—\s]+")]

# Remove meaningless short texts
meaningless = {
    "ok", "okay", "yeah", "yep", "no", "nah", "hmm", "uh", "um",
    "right", "sure", "mm", "mmm", "ha", "haha", "yes", "wow", "oh"
}
df = df[~df["text"].str.lower().isin(meaningless)]
df = df[df["text"].str.len() >= 10]

output_final = os.path.join(PROJECT_DIR, "sporc_final_clean_min.csv")
df.to_csv(output_final, index=False)
print("Final cleaned dataset saved:", output_final)

## 2. Two-Stage Summarization Pipeline

We implement a two-stage extractive–abstractive summarization framework.<br>
The extractor selects salient utterances using pseudo-label supervision, and the refiner generates abstractive summaries using a BART-based model.

### Step 9: SPoRC Dataset Class

The `SPoRCDataset` class loads the turn-level CSV and episode-level summaries, organizes utterances by episode, and provides utilities for encoding episodes and generating pseudo-labels. <br>
It supports role-specific processing, distinguishing between host, guest, and global summaries, enabling structured access to the data for both extractor and refiner stages.

---

#### SPoRCDataset: Public Methods

Exposes the main methods used by the rest of the pipeline to access episode data, generate model inputs, and create pseudo-labels.

- `episode_ids()` — Returns a list of all episode IDs.
- `get_summary(episode, summary_type)` — Returns the global, host, or guest summary for a given episode.
- `encode_episode(episode, max_utt_len)` — Tokenizes and encodes all utterances in an episode for model input.
- `build_pseudo_labels(episode, summary_type, topk)` — Generates pseudo-labels by selecting top-k utterances most similar to the reference summary.

In [ ]:
class SPoRCDataset:
    """
    Dataset layer for SPoRC.

    Responsibilities:
    1. Load turn-level CSV and group by episode
    2. Load episode-level summaries from JSONL
    3. Provide role-aware pseudo-labels for extractor training
    4. Enforce training rules:
       - global_summary MUST exist, otherwise skip episode
       - host / guest trained only if corresponding summary exists
    """

    BOILERPLATE_PATTERNS = [
        r"(?i)\bno guest\b.*?(?:\.\s*|$)",
        r"(?i)\bguest was not present\b.*?(?:\.\s*|$)",
        r"(?i)\bthere were no guests?\b.*?(?:\.\s*|$)",
        r"(?i)\bno guests?\b.*?(?:\.\s*|$)",
        r"(?i)\bno guest content\b.*?(?:\.\s*|$)",
        r"(?i)\bno guest was present in this transcript\b.*?(?:\.\s*|$)",
    ]

    def __init__(
        self,
        csv_path: str,
        summary_jsonl_path: str,
        bart_name: str = "facebook/bart-large",
        sent_emb_name: str = "all-mpnet-base-v2",
        device = "cpu",
    ):
        # ---- load CSV ----
        self.df = pd.read_csv(csv_path)
        self.episodes = self._group_by_episode(self.df)

        # ---- tokenizer (used later by models) ----
        self.tokenizer = AutoTokenizer.from_pretrained(bart_name)

        # ---- sentence embedder (for pseudo-labels) ----
        self.sent_embedder = SentenceTransformer(sent_emb_name).to(device)

        # ---- precompute whether each episode actually has host/guest turns (to avoid training a “no-guest template”). ----
        self.role_present = self._build_role_presence()

        # ---- load summaries ----
        self.summaries = {}
        self._load_summaries_jsonl(summary_jsonl_path)

    @staticmethod
    def _safe_str(x) -> str:
        if x is None:
            return ""
        try:
            if pd.isna(x):
                return ""
        except Exception:
            pass
        return str(x)

    @classmethod
    def _strip_boilerplate(cls, s: str) -> str:
        if not isinstance(s, str):
            return ""
        out = s.strip()
        for pat in cls.BOILERPLATE_PATTERNS:
            out = re.sub(pat, "", out).strip()
        # 再把多余空白收一下
        out = re.sub(r"\s+", " ", out).strip()
        return out

    @staticmethod
    def _normalize_summary(x) -> str:
        if x is None:
            s = ""
        elif isinstance(x, str):
            s = x.strip()
        elif isinstance(x, dict):
            for k in ("summary", "text", "content", "output"):
                v = x.get(k)
                if isinstance(v, str):
                    s = v.strip()
                    break
            else:
                s = json.dumps(x, ensure_ascii=False).strip()
        elif isinstance(x, list):
            parts = [SPoRCDataset._normalize_summary(i) for i in x]
            s = " ".join([p for p in parts if p]).strip()
        else:
            s = str(x).strip()

        # 过滤常见占位符
        if s.upper() in {"N/A", "NA", "NULL", "NONE"}:
            return ""
        return s


    # ------------------------------------------------------------------
    # internal helpers
    # ------------------------------------------------------------------

    def _group_by_episode(self, df: pd.DataFrame):
        episodes = defaultdict(list)

        for _, row in df.iterrows():
            ep = self._safe_str(row.get("episode", ""))
            try:
                turn = int(row.get("turn", None))
            except Exception:
                continue
            episodes[ep].append(
                {
                    "turn": turn,
                    "role": self._safe_str(row.get("role", "")),
                    "speaker": self._safe_str(row.get("speaker", "")),
                    "text": self._safe_str(row.get("text", "")),
                }
            )

        for ep in episodes:
            episodes[ep] = sorted(episodes[ep], key=lambda x: x["turn"])

        return episodes

    def _build_role_presence(self):
        role_present = {}
        for ep, utts in self.episodes.items():
            has_host = any(u.get("role") == "host" and str(u.get("text","")).strip() != "" for u in utts)
            has_guest = any(u.get("role") == "guest" and str(u.get("text","")).strip() != "" for u in utts)
            role_present[ep] = {"host": has_host, "guest": has_guest}
        return role_present

    def _load_summaries_jsonl(self, path: str):
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                obj = json.loads(line)
                ep = obj.get("episode", None)
                if ep is None:
                    continue
                ep = str(ep)

                g = self._normalize_summary(obj.get("global_summary"))
                h = self._normalize_summary(obj.get("host_summary"))
                ge = self._normalize_summary(obj.get("guest_summary"))

                # Remove "no-guest template sentence."
                g = self._strip_boilerplate(g)
                h = self._strip_boilerplate(h)
                ge = self._strip_boilerplate(ge)

                # If this episode has no guest turns at all, forcibly set guest_summary to empty (do not train, do not evaluate).
                if ep in self.role_present and not self.role_present[ep]["guest"]:
                    ge = ""

                self.summaries[ep] = {"global": g, "host": h, "guest": ge}

    # ------------------------------------------------------------------
    # public API
    # ------------------------------------------------------------------

    def episode_ids(self):
        return list(self.episodes.keys())

    def get_summary(self, episode: str, summary_type: str):
        if episode not in self.summaries:
            return ""
        return self.summaries[episode].get(summary_type, "")

    def encode_episode(self, episode: str, max_utt_len: int = 64):
        utts = self.episodes[episode]
        texts = [self._safe_str(u.get("text", "")) for u in utts]
        roles = [self._safe_str(u.get("role", "")) for u in utts]

        enc = self.tokenizer(
            texts,
            truncation=True,
            padding="max_length",
            max_length=max_utt_len,
            return_tensors="pt",
        )

        return enc["input_ids"], enc["attention_mask"], roles, texts

    def build_pseudo_labels(self, episode: str, summary_type: str = "global", topk: int = 5):
        # ---- episode must have summaries ----
        if episode not in self.summaries:
            print(f"[WARN] episode {episode}: summary not found, skip training")
            return None

        summaries = self.summaries[episode]

        # ---- global summary is mandatory ----
        if summaries.get("global", "").strip() == "":
            print(
                f"[WARN] episode {episode}: global_summary missing, skip training"
            )
            return None

        # ---- role-specific summary ----
        summary_text = summaries.get(summary_type, "")
        if summary_text is None or summary_text.strip() == "":
            return None
        summary_text = summary_text.strip()

        # ---- select utterances ----
        utts = self.episodes.get(episode, [])
        if not utts:
            return None

        if summary_type == "host":
            idx_text = [(i, self._safe_str(u.get("text", "")))
                        for i, u in enumerate(utts) if self._safe_str(u.get("role","")) == "host"]
        elif summary_type == "guest":
            idx_text = [(i, self._safe_str(u.get("text", "")))
                        for i, u in enumerate(utts) if self._safe_str(u.get("role","")) == "guest"]
        else:
            idx_text = [(i, self._safe_str(u.get("text", ""))) for i, u in enumerate(utts)]

        idx_text = [(i, t) for i, t in idx_text if t.strip() != ""]
        if len(idx_text) == 0:
            return None

        indices, texts = zip(*idx_text)

        # ---- sentence similarity ----
        utt_emb = self.sent_embedder.encode(list(texts), convert_to_tensor=True)
        sum_emb = self.sent_embedder.encode(summary_text, convert_to_tensor=True)
        if sum_emb.dim() == 1:
            sum_emb = sum_emb.unsqueeze(0)

        sims = util.cos_sim(utt_emb, sum_emb).squeeze(1)
        k = min(topk, sims.size(0))
        topk_local = torch.topk(sims, k=k).indices.tolist()

        return [indices[i] for i in topk_local]

### Step 10: Model Definition

The two-stage summarization model separates extraction from generation. `UtteranceEncoder` encodes utterances, `Extractor` predicts importance scores for selection, and Refiner produces abstractive summaries using BART. <br>
`TwoStageSummarizer` integrates these components into a unified pipeline.


In [ ]:
class UtteranceEncoder(nn.Module):
    """
    Encode each utterance into a fixed-size vector
    using a pretrained BART encoder.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.encoder = AutoModel.from_pretrained(bart_name).encoder
        self.encoder.to(device)

    def forward(self, input_ids, attention_mask):
        """
        input_ids:      [num_utts, seq_len]
        attention_mask: [num_utts, seq_len]

        returns:
            utt_emb: [num_utts, hidden_dim]
        """
        outputs = self.encoder(
            input_ids=input_ids,
            attention_mask=attention_mask,
            return_dict=True
        )

        # Use the first token of each sentence（<s>）as the sentence vector.
        utt_emb = outputs.last_hidden_state[:, 0]
        return utt_emb


class Extractor(nn.Module):
    """
    Simple extractor:
    Given utterance embeddings, predict an importance score
    for each utterance.
    """
    def __init__(self, hidden_dim):
        super().__init__()
        self.classifier = nn.Linear(hidden_dim, 1)

    def forward(self, utt_emb):
        """
        utt_emb: [num_utts, hidden_dim]

        returns:
            logits: [num_utts]
        """
        logits = self.classifier(utt_emb).squeeze(-1)
        return logits


class Refiner(nn.Module):
    """
    Abstractive summarizer based on BART.
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()
        self.model = BartForConditionalGeneration.from_pretrained(bart_name)

    def forward(self, input_ids, attention_mask, labels=None):
        """
        Standard BART forward.

        If labels is provided, returns training loss.
        """
        return self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            labels=labels,
            return_dict=True
        )


class TwoStageSummarizer(nn.Module):
    """
    Two-stage summarization model:
    1) UtteranceEncoder + Extractor (sentence selection)
    2) Refiner (BART generation)
    """
    def __init__(self, bart_name="facebook/bart-large"):
        super().__init__()

        self.encoder = UtteranceEncoder(bart_name)
        self.extractor = Extractor(hidden_dim=1024)  # bart-large hidden size
        self.refiner = Refiner(bart_name)

    def forward_extractor(self, input_ids, attention_mask):
        """
        Forward pass for extractor only.

        returns:
            logits:  [num_utts]
            utt_emb: [num_utts, hidden_dim]
        """
        utt_emb = self.encoder(input_ids, attention_mask)
        logits = self.extractor(utt_emb)
        return logits, utt_emb

### Step 11: Training Utilities
This section defines helper functions used during model training.  <br>
The extractor is trained using pseudo-labels derived from reference summaries, while the refiner is trained in a supervised sequence-to-sequence setting.

In [ ]:
def train_extractor(
    model,
    dataset,
    episodes,
    summary_type="global",
    topk=5,
    lr=1e-4,
):
    model.encoder.eval()      # encoder frozen
    model.extractor.train()

    optimizer = torch.optim.AdamW(model.extractor.parameters(), lr=lr)
    criterion = torch.nn.BCEWithLogitsLoss()

    total_loss = 0.0
    count = 0

    for ep in episodes:
        pos_indices = dataset.build_pseudo_labels(
            ep, summary_type=summary_type, topk=topk
        )
        if pos_indices is None:
            continue

        input_ids, attn, _, _ = dataset.encode_episode(ep)
        input_ids = input_ids.to(device)
        attn = attn.to(device)

        with torch.no_grad():
            utt_emb = model.encoder(input_ids, attn)

        logits = model.extractor(utt_emb)
        labels = torch.zeros_like(logits)
        labels[pos_indices] = 1.0

        loss = criterion(logits, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        count += 1

    return total_loss / max(count, 1)

### Step 12: Inference Utilities

`generate_summary` produces summaries for an episode by first selecting key utterances with the extractor and then generating fluent text with the refiner.<br>
Supports global and role-specific summaries.

In [ ]:
SUMMARY_PREFIX = {
    "global": "<GLOBAL>",
    "host": "<HOST>",
    "guest": "<GUEST>",
}

def generate_summary(
    model,
    dataset,
    episode,
    summary_type="global",
    max_sentences=6,
    max_len=256,
):
    model.eval()

    input_ids, attn, roles, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    # If the role doesn’t exist at all, return empty directly (don’t let the model hallucinate a “no guest”).
    if summary_type in ("host", "guest"):
        if not any(r == summary_type for r in roles):
            return ""

    # ----- extractor: select sentences -----
    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)
        if summary_type in ("host", "guest"):
            keep = torch.tensor([r == summary_type for r in roles], device=logits.device)
            logits = logits.masked_fill(~keep, -1e9)

        k = min(max_sentences, logits.size(0))
        topk = torch.topk(logits, k=k).indices

    selected = [texts[i] for i in sorted(topk.tolist()) if texts[i].strip() != ""]
    if len(selected) == 0:
        return ""

    # ----- refiner: generate summary -----
    tok = dataset.tokenizer
    concat_text = SUMMARY_PREFIX[summary_type] + " " + " ".join(selected)

    enc = tok(concat_text, truncation=True, max_length=max_len, return_tensors="pt").to(device)

    with torch.no_grad():
        out = model.refiner.model.generate(
            **enc,
            max_length=150,
            num_beams=4,
        )

    return tok.decode(out[0], skip_special_tokens=True).strip()

In [ ]:
def train_refiner_on_episode(
    model,
    dataset,
    episode,
    summary_type,
    optimizer,
    max_len=256,
):
    ref = dataset.get_summary(episode, summary_type)
    if ref is None or ref.strip() == "":
        return None

    input_ids, attn, _, texts = dataset.encode_episode(episode)
    input_ids = input_ids.to(device)
    attn = attn.to(device)

    with torch.no_grad():
        logits, _ = model.forward_extractor(input_ids, attn)
        topk = torch.topk(logits, k=min(6, logits.size(0))).indices

    selected = [texts[i] for i in sorted(topk.tolist())]
    prefix = SUMMARY_PREFIX[summary_type]
    concat_text = prefix + " " + " ".join(selected)

    tok = dataset.tokenizer
    enc = tok(
        concat_text,
        truncation=True,
        max_length=max_len,
        return_tensors="pt",
    ).to(device)

    tgt = tok(
        ref,
        truncation=True,
        max_length=150,
        return_tensors="pt",
    ).to(device)

    model.refiner.train()
    out = model.refiner(
        input_ids=enc["input_ids"],
        attention_mask=enc["attention_mask"],
        labels=tgt["input_ids"],
    )

    loss = out.loss
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    return loss.item()

### Step 13: Evaluation

Evaluates summaries using ROUGE-L to measure overlap with reference summaries, supporting global, host, or guest evaluations. <br>


In [ ]:
# Initialize ROUGE scorer
rouge_scorer_l = rouge_scorer.RougeScorer(
    ["rougeL"], use_stemmer=True
)

def compute_rouge_l(pred, ref):
    """
    Compute ROUGE-L F1 score between a generated summary and a reference summary.
    The reference summary is treated as a silver (LLM-generated) gold.
    """
    if ref is None or ref.strip() == "":
        return None
    score = rouge_scorer_l.score(ref, pred)
    return score["rougeL"].fmeasure

### Step 14: Main Training & Generation

This section orchestrates the full pipeline: dataset and model setup, adding role control tokens, preparing training episodes, training the extractor and refiner, and generating example summaries for a sample episode.


In [ ]:
# -------- 0. Setup --------
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# -------- 1. Build Dataset --------
dataset = SPoRCDataset(
    csv_path="sporc_turns_selected_clean.csv",
    summary_jsonl_path="sporc_full_summaries_7b.jsonl",
    device=device,
)

# -------- 2. Build Model --------
model = TwoStageSummarizer().to(device)

# -------- 3. Add control tokens for Refiner (run once, before training) --------
SUMMARY_PREFIX = {
    "global": "<GLOBAL>",
    "host": "<HOST>",
    "guest": "<GUEST>",
}

dataset.tokenizer.add_tokens(["<GLOBAL>", "<HOST>", "<GUEST>"])
model.refiner.resize_token_embeddings(len(dataset.tokenizer))

# -------- 4. Prepare training episodes (start with 250 episodes) --------
episodes_with_summary = list(dataset.summaries.keys())
train_episodes = episodes_with_summary[:250]
test_episodes  = episodes_with_summary[250:300]

print(f"Training episodes: {len(train_episodes)}")
print(f"Test episodes : {len(test_episodes)}")

# =====================================================
# Part A: Train Extractor (three summary types)
# =====================================================
print("\n===== Train Extractor =====")

for st in ["global", "host", "guest"]:
    loss = train_extractor(
        model,
        dataset,
        train_episodes,
        summary_type=st,
    )
    print(f"[Extractor] {st} average loss: {loss:.4f}")

# =====================================================
# Part B: Train Refiner (supervised, three summary types)
# =====================================================
print("\n===== Train Refiner =====")

# IMPORTANT: create the optimizer only once
refiner_optimizer = torch.optim.AdamW(
    model.refiner.parameters(), lr=2e-5
)

for st in ["global", "host", "guest"]:
    losses = []
    for ep in train_episodes:
        loss = train_refiner_on_episode(
            model,
            dataset,
            ep,
            summary_type=st,
            optimizer=refiner_optimizer,
        )
        if loss is not None:
            losses.append(loss)

    if len(losses) > 0:
        print(f"[Refiner] {st} average loss: {sum(losses)/len(losses):.4f}")
    else:
        print(f"[Refiner] {st}: no valid samples")

# =================================================
# Part C: Test Refiner with ROUGE-L
# =================================================
print("\n===== Evaluate Refiner (ROUGE-L) =====")

rouge_scores = {
    "global": [],
    "host": [],
    "guest": [],
}

for ep in test_episodes:
    for st in ["global", "host", "guest"]:
        # Generate summary
        pred = generate_summary(
            model, dataset, ep, summary_type=st
        )

        # Silver reference summary (LLM-generated)
        gold = dataset.get_summary(ep, st)

        score = compute_rouge_l(pred, gold)
        if score is not None:
            rouge_scores[st].append(score)

# Report average ROUGE-L
for st in ["global", "host", "guest"]:
    if len(rouge_scores[st]) > 0:
        avg = sum(rouge_scores[st]) / len(rouge_scores[st])
        print(f"ROUGE-L ({st}): {avg:.4f}")
    else:
        print(f"ROUGE-L ({st}): N/A")

# =====================================================
# Part D: Generate summaries (for QA / Conflict Detection)
# =====================================================
print("\n===== Generate Summaries =====")

# Use a single episode as an example
ep = train_episodes[0]

global_summary = generate_summary(
    model, dataset, ep, summary_type="global"
)
host_summary = generate_summary(
    model, dataset, ep, summary_type="host"
)
guest_summary = generate_summary(
    model, dataset, ep, summary_type="guest"
)

print("\n--- GLOBAL SUMMARY ---")
print(global_summary)

print("\n--- HOST SUMMARY ---")
print(host_summary)

print("\n--- GUEST SUMMARY ---")
print(guest_summary)

### Step 15: Load MNLI Model

Loads a pre-trained NLI model to detect contradictions between host and guest summaries. <br>
Provides a helper function `nli_predict` for computing label, confidence, and probabilities.

In [ ]:
nli_model_name = "microsoft/deberta-v3-large-mnli"
nli_tokenizer = AutoTokenizer.from_pretrained(nli_model_name)
nli_model = AutoModelForSequenceClassification.from_pretrained(nli_model_name).to(device)
nli_model.eval()

LABEL_MAP = {0: "CONTRADICTION", 1: "NEUTRAL", 2: "ENTAILMENT"}

@torch.no_grad()
def nli_predict(premise, hypothesis):
    inputs = nli_tokenizer(
        premise,
        hypothesis,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    logits = nli_model(**inputs).logits
    probs = F.softmax(logits, dim=-1)[0]
    label_id = torch.argmax(probs).item()

    return {
        "label": LABEL_MAP[label_id],
        "score": probs[label_id].item(),
        "probs": {
            LABEL_MAP[i]: probs[i].item() for i in range(3)
        }
    }

### Step 16: Local Conflict Evidence Extraction

Identifies sentence-level contradictions between host and guest summaries by computing sentence embeddings, matching semantically similar sentences, and applying the NLI model to detect conflicts.

In [ ]:
try:
    nltk.data.find("tokenizers/punkt")
except LookupError:
    nltk.download("punkt")

sent_encoder = SentenceTransformer("all-MiniLM-L6-v2").to(device)

def split_sentences(text):
    return nltk.sent_tokenize(text)

@torch.no_grad()
def find_conflict_evidence(host_summary, guest_summary, topk=3):
    host_sents = split_sentences(host_summary)
    guest_sents = split_sentences(guest_summary)

    if len(host_sents) == 0 or len(guest_sents) == 0:
        return []

    host_emb = sent_encoder.encode(host_sents, convert_to_tensor=True)
    guest_emb = sent_encoder.encode(guest_sents, convert_to_tensor=True)

    sim = util.cos_sim(host_emb, guest_emb)
    pairs = []

    # top-k pair
    for i in range(min(topk, sim.numel())):
        idx = torch.argmax(sim)
        h, g = divmod(idx.item(), sim.size(1))
        pairs.append((host_sents[h], guest_sents[g]))
        sim[h, g] = -1  # mask

    evidence = []
    for h_sent, g_sent in pairs:
        nli_res = nli_predict(h_sent, g_sent)
        if nli_res["label"] == "CONTRADICTION":
            evidence.append({
                "host_sentence": h_sent,
                "guest_sentence": g_sent,
                "confidence": nli_res["score"]
            })

    return evidence

### Step 17: Build Structured Conflict Output

Combines overall NLI classification with supporting evidence sentence pairs to produce a structured conflict report per episode, indicating stance disagreements and highlighting evidence.

In [ ]:
def build_conflict_output(episode, host_summary, guest_summary):
    overall = nli_predict(host_summary, guest_summary)
    evidence = find_conflict_evidence(host_summary, guest_summary)

    return {
        "episode": episode,
        "conflict_overall": {
            "label": overall["label"],
            "score": overall["score"]
        },
        "conflict_type": (
            "stance_disagreement"
            if overall["label"] == "CONTRADICTION"
            else "none"
        ),
        "evidence": evidence
    }

### Step 18: Role-Aware Q&A

Implements a question-answering interface over summaries and conflicts.

In [ ]:
def answer_question(question, episode_result):
    q = question.lower()

    summaries = episode_result["summaries"]
    conflict = episode_result["conflict"]

    # ---- Conflict existence ----
    if "disagree" in q or "conflict" in q:
        if conflict["label"] == "CONTRADICTION":
            return "Yes. A disagreement between the host and the guest was detected."
        else:
            return "No clear disagreement between the host and the guest was detected."

    # ---- Conflict details ----
    if "what" in q and "disagree" in q:
        if conflict["label"] != "CONTRADICTION":
            return "No specific disagreement was identified in this episode."

        evidence = conflict.get("evidence", [])
        if len(evidence) == 0:
            return "A disagreement was detected, but no specific evidence sentence pair was identified."

        e = evidence[0]
        return (
            "The disagreement is reflected in the following statements:\n"
            f"- Host: {e['host_sentence']}\n"
            f"- Guest: {e['guest_sentence']}"
        )

    # ---- Role-specific viewpoints ----
    if "host" in q and ("view" in q or "opinion" in q):
        return summaries["host"]

    if "guest" in q and ("view" in q or "opinion" in q):
        return summaries["guest"]

    # ---- Global summary ----
    if "summary" in q or "about" in q:
        return summaries["global"]

    # ---- Fallback ----
    return "This question is not supported by the current QA system."

### Step 19: Episode-Level Summarization

Generates global, host, and guest summaries for a given episode and performs conflict detection if both host and guest summaries exist. <br>
Returns a dictionary containing episode ID, role-aware summaries, and conflict information.

In [ ]:
def run_episode(model, dataset, episode):
    # 1) summaries
    g = generate_summary(model, dataset, episode, "global")
    h = generate_summary(model, dataset, episode, "host")
    ge = generate_summary(model, dataset, episode, "guest")

    # 2) conflict (If either host or guest is empty, it is considered indeterminable)
    conflict = {"label": "NEUTRAL", "score": 0.0, "evidence": []}
    if (h or "").strip() != "" and (ge or "").strip() != "":
        conflict_obj = build_conflict_output(episode, h, ge)
        conflict = {
            "label": conflict_obj["conflict_overall"]["label"],
            "score": conflict_obj["conflict_overall"]["score"],
            "evidence": conflict_obj["evidence"],
        }

    return {
        "episode": episode,
        "summaries": {"global": g, "host": h, "guest": ge},
        "conflict": conflict
    }

### Step 20: Quick Sanity Check

A test run of the system up till this point, just to make sure it's working as expected.

In [ ]:
# Quick test of the system
ep = test_episodes[0]
episode_result = run_episode(model, dataset, ep)
episode_result

As the system seems to be working correctly, we continue unto the next step to ensure that we can run the system on a larger scale.

### Step 21: Saving and loading JSON

Runs the summarization and conflict detection over a list of episodes and saves the results to a JSONL file. <br>
As well as loads previously saved episode results from a JSONL file.

In [ ]:
# Saves to JSON
def save_results_jsonl(model, dataset, episodes, output_path="test_results.jsonl"):
    with open(output_path, "w", encoding="utf-8") as f:
        for ep in tqdm(episodes, desc="Processing episodes"):
            result = run_episode(model, dataset, ep)
            f.write(json.dumps(result, ensure_ascii=False) + "\n")

    print("Saved:", output_path)

# Load JSON
def load_jsonl(path):
    data = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            data.append(json.loads(line))
    return data

### Step 22: Testrun of the System

Runs a complete test of the system on a list of episodes, generating summaries, detecting conflicts, and computing simple summary statistics. <br>

We ask five questions:
1. What is this episode about? (Q1)
2. What is the host view? (Q2)
3. What is the guest view? (Q3)
4. Do they disagree? (Q4)
5. What do they disagree about? (Q5)

In [ ]:
# Run and save all test episodes
save_results_jsonl(model, dataset, test_episodes, output_path="test_results.jsonl")

# Load results
results = load_jsonl("test_results.jsonl")

# Inspect a single episode - First episode as example (results[0])
example_result = results[0]
print("Episode:", example_result["episode"])
print("\nQ1:", answer_question("What is this episode about?", example_result))
print("\nQ2:", answer_question("What is the host view?", example_result))
print("\nQ3:", answer_question("What is the guest view?", example_result))
print("\nQ4:", answer_question("Do they disagree?", example_result))
print("\nQ5:", answer_question("What do they disagree about?", example_result))

# Compute summary statistics
has_guest = sum(1 for r in results if (r["summaries"].get("guest","") or "").strip() != "")
has_conflict = sum(1 for r in results if r["conflict"].get("label") == "CONTRADICTION")

print("Total episodes processed:", len(results))
print("Episodes with guest summary:", has_guest)
print("Episodes with detected contradiction:", has_conflict)